# Problema 1: El Espejismo Operativo

Trabajo con el archivo delivery_logs_parcial.csv. El VP dice que Bici tiene menos reclamos que Moto y propone cambiar flota. Sospecho que es un sesgo por Zona y Distancia.


In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Invitado 1\OneDrive\Documentos\UNIVERSIDAD EAFIT\LINEA_DE_ENFASIS\Estadistica Aplicada\Examen01_EstApl\delivery_logs_parcial.csv')

print(df.shape)
print(df['vehicle_type'].value_counts())


(5000, 7)
vehicle_type
Moto    3336
Bici    1664
Name: count, dtype: int64


### 1. El argumento del VP — Probabilidad condicional simple

Primero miro la proporción de reclamos dentro de cada tipo de vehículo sin condicionar nada más. Esto es lo que vio el VP.


In [2]:
p_reclamo_bici = df[df['vehicle_type']=='Bici']['claim_filed'].mean()
p_reclamo_moto = df[df['vehicle_type']=='Moto']['claim_filed'].mean()

print('P[Reclamo | Bici] =', p_reclamo_bici)
print('P[Reclamo | Moto] =', p_reclamo_moto)


P[Reclamo | Bici] = 0.18389423076923078
P[Reclamo | Moto] = 0.2511990407673861


**Análisis 1.** A nivel global el VP tiene razón en los números. Bici ~18.4% de reclamos frente a ~25.1% en moto. Esa diferencia justifica su intuición si se mira solo el vehículo, pero mezcla zonas y distancias.


### 2. Desmontando el espejismo — Probabilidad total con la zona

Veo la distribución por zona para confirmar el sesgo de operación.


In [3]:
print(pd.crosstab(df['zone'], df['vehicle_type']))

centro = df[df['zone']=='Centro']

p_reclamo_moto_centro = centro[centro['vehicle_type']=='Moto']['claim_filed'].mean()
p_reclamo_bici_centro = centro[centro['vehicle_type']=='Bici']['claim_filed'].mean()

print('P[Reclamo | Moto ∩ Centro] =', p_reclamo_moto_centro)
print('P[Reclamo | Bici ∩ Centro] =', p_reclamo_bici_centro)


vehicle_type  Bici  Moto
zone                    
Centro        1405  1095
Norte          160  1341
Sur             99   900
P[Reclamo | Moto ∩ Centro] = 0.08310502283105023
P[Reclamo | Bici ∩ Centro] = 0.12740213523131672


**Análisis 2.** Cuando aislo la zona Centro, el espejismo se invierte. La moto pasa a tener menos reclamos que la bici en la misma zona: 8.3% vs 12.7%. La ventaja aparente de la bici a nivel global no es por el vehículo, sino por dónde suele operar. La zona está confundiendo el resultado.


### 3. Auditoría inversa — Teorema de Bayes

Ahora la pregunta práctica: llega un reclamo en el Centro, ¿qué tan probable es que haya sido en moto?


In [4]:
centro = df[df['zone']=='Centro']
reclamo_centro = centro[centro['claim_filed']==1]

p_moto_dado_reclamo_centro = (reclamo_centro['vehicle_type']=='Moto').mean()

print('P[Moto | Reclamo ∩ Centro] =', p_moto_dado_reclamo_centro)
print('Reclamos en Centro:', len(reclamo_centro))
print('De ellos en Moto:', (reclamo_centro['vehicle_type']=='Moto').sum())


P[Moto | Reclamo ∩ Centro] = 0.337037037037037
Reclamos en Centro: 270
De ellos en Moto: 91


**Análisis 3.** Aunque la moto tiene menor tasa de reclamo dentro del Centro, representa alrededor del 33.7% de los reclamos que ocurren en esa zona. El cálculo de Bayes permite ver la contribución real de cada vehículo al problema.

### Conclusión del punto 1

El argumento del VP se sostiene solo en el agregado. Al condicionar por zona, la moto resulta mejor en Centro. La conclusión operativa no es cambiar flota de moto por bici de forma indiscriminada, sino revisar por qué la bici genera más reclamos en la misma zona donde debería ser más eficiente.
